# 模型对比分析报告

本Notebook汇总三个CNN模型（MiniCNN / VGGLite / MicroResNet）的训练结果，进行系统性对比。

前提：确保 02/03/04 三个训练Notebook已经运行完成，checkpoint和日志已生成。

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml

from training.trainer import PROJECT_ROOT
from data.dataloader import FER2013Dataset

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

config_path = PROJECT_ROOT / 'configs' / 'training_config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

CLASS_NAMES = config['data']['class_names']
MODEL_NAMES = ['mini_cnn', 'vgg_lite', 'micro_resnet']

# 加载训练历史
histories = {}
for name in MODEL_NAMES:
    hist_path = PROJECT_ROOT / 'training' / 'logs' / name / 'history.json'
    if hist_path.exists():
        with open(hist_path) as f:
            histories[name] = json.load(f)
        print(f"已加载 {name} 训练历史 ({len(histories[name]['train_loss'])} epochs)")
    else:
        print(f"警告: {name} 训练历史不存在，请先运行对应训练Notebook")

## 1. 训练曲线对比

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {'mini_cnn': 'blue', 'vgg_lite': 'green', 'micro_resnet': 'red'}

for name, hist in histories.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    c = colors[name]
    axes[0, 0].plot(epochs, hist['train_loss'], color=c, label=f'{name}', linestyle='-')
    axes[0, 1].plot(epochs, hist['val_loss'], color=c, label=f'{name}', linestyle='-')
    axes[1, 0].plot(epochs, hist['train_acc'], color=c, label=f'{name}', linestyle='-')
    axes[1, 1].plot(epochs, hist['val_acc'], color=c, label=f'{name}', linestyle='-')

axes[0, 0].set_title('Train Loss'); axes[0, 0].legend()
axes[0, 1].set_title('Val Loss'); axes[0, 1].legend()
axes[1, 0].set_title('Train Accuracy'); axes[1, 0].legend()
axes[1, 1].set_title('Val Accuracy'); axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xlabel('Epoch')

plt.suptitle('三模型训练曲线对比', fontsize=14)
plt.tight_layout()
curve_dir = PROJECT_ROOT / 'analysis' / 'training_curves'
curve_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(curve_dir / 'comparison_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 最优性能汇总

In [ ]:
# 汇总表
summary = []
for name in MODEL_NAMES:
    if name not in histories:
        continue
    hist = histories[name]
    best_epoch = np.argmax(hist['val_acc']) + 1
    summary.append({
        '模型': name,
        '参数量': {'mini_cnn': '~50K', 'vgg_lite': '~1.5M', 'micro_resnet': '~400K'}[name],
        '最优Epoch': best_epoch,
        '最优Val Acc': f"{hist['val_acc'][best_epoch-1]:.4f}",
        '最优Val Loss': f"{hist['val_loss'][best_epoch-1]:.4f}",
        '最终Train Acc': f"{hist['train_acc'][-1]:.4f}",
        '总Epochs': len(hist['train_loss']),
    })

summary_df = pd.DataFrame(summary)
print("=== 模型性能汇总 ===")
summary_df

## 3. 混淆矩阵对比

In [ ]:
# 加载保存的混淆矩阵图片
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for i, name in enumerate(MODEL_NAMES):
    cm_path = PROJECT_ROOT / 'analysis' / 'confusion_matrices' / f'{name}_cm.png'
    if cm_path.exists():
        img = plt.imread(str(cm_path))
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(name)
    else:
        axes[i].text(0.5, 0.5, f'{name}\n(未生成)', ha='center', va='center', fontsize=14)
        axes[i].axis('off')

plt.suptitle('混淆矩阵对比', fontsize=14)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'comparison_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 各类别准确率对比

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from inference.infer_utils import get_model_class

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 加载测试集
df = pd.read_csv(PROJECT_ROOT / 'data' / 'fer2013.csv')
if 'Usage' in df.columns:
    test_df = df[df['Usage'] == 'PrivateTest'].reset_index(drop=True)
else:
    test_df = df  # fallback

test_dataset = FER2013Dataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

# 逐模型推理并收集 per-class recall
per_class_results = {}

for name in MODEL_NAMES:
    ckpt_path = PROJECT_ROOT / 'training' / 'checkpoints' / name / 'global_best.pth'
    if not ckpt_path.exists():
        print(f'跳过 {name}: checkpoint 不存在')
        continue

    ModelClass = get_model_class(name)
    model = ModelClass(num_classes=len(CLASS_NAMES)).to(device)
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True)
    per_class_results[name] = {cls: report[cls]['recall'] for cls in CLASS_NAMES}
    print(f'{name} 测试完成')

# 绘制分组柱状图
if per_class_results:
    x = np.arange(len(CLASS_NAMES))
    width = 0.25
    model_colors = {'mini_cnn': '#3498db', 'vgg_lite': '#2ecc71', 'micro_resnet': '#e74c3c'}

    fig, ax = plt.subplots(figsize=(12, 6))

    for i, (name, recalls) in enumerate(per_class_results.items()):
        values = [recalls[cls] for cls in CLASS_NAMES]
        bars = ax.bar(x + i * width, values, width, label=name, color=model_colors.get(name, None), alpha=0.85)
        # 在柱顶标注数值
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('情感类别')
    ax.set_ylabel('Recall')
    ax.set_title('各类别 Recall 对比（测试集）')
    ax.set_xticks(x + width)
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'comparison_per_class_recall.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('无可用模型结果，跳过各类别准确率对比图。')

## 5. 参数效率分析

In [ ]:
# 参数量 vs 准确率 散点图
param_counts = {'mini_cnn': 50_000, 'vgg_lite': 1_500_000, 'micro_resnet': 400_000}

fig, ax = plt.subplots(figsize=(8, 6))
for name in MODEL_NAMES:
    if name not in histories:
        continue
    best_acc = max(histories[name]['val_acc'])
    ax.scatter(param_counts[name], best_acc, s=200, zorder=5)
    ax.annotate(name, (param_counts[name], best_acc),
                textcoords='offset points', xytext=(10, 10), fontsize=12)

ax.set_xlabel('参数量')
ax.set_ylabel('最优验证准确率')
ax.set_title('参数效率：参数量 vs 准确率')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'comparison_param_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 结论与最优模型选择

根据以上对比分析，记录：
- [ ] 哪个模型在准确率上最优
- [ ] 哪个模型在参数效率上最优
- [ ] 各模型在哪些类别上表现差异显著
- [ ] 最优模型是否应部署到推理应用

In [ ]:
# 自动选择最优模型并复制到推理目录
import shutil

if histories:
    best_model_name = max(histories.keys(), key=lambda n: max(histories[n]['val_acc']))
    best_acc = max(histories[best_model_name]['val_acc'])
    print(f'最优模型: {best_model_name} (Val Acc: {best_acc:.4f})')

    # 使用 global_best.pth（双文件 checkpoint 机制下的跨会话最佳）
    src = PROJECT_ROOT / 'training' / 'checkpoints' / best_model_name / 'global_best.pth'
    dst = PROJECT_ROOT / 'inference' / 'saved_models' / f'{best_model_name}_best.pth'
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f'已自动复制最优模型到推理目录: {dst}')
    elif dst.exists():
        print(f'推理目录已存在该模型: {dst}')
    else:
        print(f'Global best checkpoint不存在: {src}')
else:
    print('无训练历史，无法自动选择')